In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
import os

# --- Configuration ---
PROJECT_NAME = "GalaxyMorphologyClassifier"
COURSE = "AIG210-Computer Vision Final Project - W2026"
DATA_PATH = "./galaxy_zoo_data" # Path to extracted Kaggle data
BATCH_SIZE = 32
NUM_CLASSES = 3  # Spiral, Elliptical, Irregular
EPOCHS = 15
LEARNING_RATE = 0.001

# --- 1. Data Preprocessing & Augmentation ---
# Galaxies can appear at any angle, so rotation and flipping are essential.
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(90),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# Note: This assumes the dataset is organized into subfolders by class.
# GalaxyZoo 2 often requires a custom Dataset class for CSV mapping, 
# but ImageFolder is used here for standard CV classification logic.
full_dataset = datasets.ImageFolder(os.path.join(DATA_PATH, 'train'), data_transforms['train'])

# Splitting logic (80/20)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_data, val_data = torch.utils.data.random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE)

# --- 2. Model Architecture (ResNet18) ---
def get_model():
    # Load pre-trained weights to leverage low-level feature detection (edges, blobs)
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    
    # Freeze earlier layers (optional, but good for small datasets)
    # for param in model.parameters():
    #     param.requires_grad = False
    
    # Replace the final fully connected layer for 3-class morphology
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, NUM_CLASSES)
    
    return model

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = get_model().to(device)

# --- 3. Loss & Optimizer ---
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# --- 4. Training Loop ---
def train_model(model, criterion, optimizer, epochs):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss:.4f}")

# --- 5. Execution ---
if __name__ == "__main__":
    print(f"Starting {PROJECT_NAME} for {COURSE}...")
    train_model(model, criterion, optimizer, EPOCHS)
    
    # Save the model
    torch.save(model.state_dict(), "galaxy_classifier.pth")
    print("Training complete. Model saved.")

ModuleNotFoundError: No module named 'torch'